In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math as m
from scipy import signal as s

import obspy
from obspy import UTCDateTime
from obspy import read
from obspy.clients.fdsn import Client

In [2]:
# string describing the date and time of the earthquake (UTC)
dtstr = "2001-02-28T18:54:32" # Nisqually

# string describing the FDSN client used to download the seismograms
fdsn_client = Client('IRIS')

# File containing station codes
stacodes = '../nisq_data/station_codes_SWIF_REG2_nisq.csv'

# frequency bounds for filtering seismograms
flo=0.05 # lower frequency bound for filtering
fhi=10.5 # upper frequency bound for filtering

# Buffer for filtration (s)
buff = 20

In [3]:
# Download stations data
codes = np.loadtxt(stacodes,dtype='str',delimiter=',',skiprows=1)
net = codes[:,1]
sta = codes[:,0]
chans = ['FXZ','FXY','FXX']
comp = "semv"

In [4]:
dt=UTCDateTime(dtstr)
for i in range(len(sta)):
    try:
        if(codes[i,1]=='UW'):
            if(i%10==0):
                    print('station #'+str(i)+' of '+str(len(sta)))
    
            chanstr=codes[i,3]+'*'
            print(codes[i,1],codes[i,0],codes[i,4],chanstr,dt)
            
            st = fdsn_client.get_waveforms(network=codes[i,1],station=codes[i,0],location='*',channel=chanstr,starttime=dt,endtime=dt+120)
            pre_filt = (0.005, 0.006, 30.0, 35.0)
            inv = fdsn_client.get_stations(
                network=codes[i,1], station=codes[i,0], location=codes[i,4], channel=chanstr,
                starttime=dt, endtime=dt+120, level='response')
            st.remove_response(inventory=inv,output='ACC', pre_filt=pre_filt)
    
            stats = st[0].stats
            delt = stats.delta
            sost = s.butter(2,[flo,fhi],'bandpass',fs=1/delt,output='sos')
            trE = s.sosfiltfilt(sost,st[0].data)
            trN = s.sosfiltfilt(sost,st[1].data)
            trZ = s.sosfiltfilt(sost,st[2].data)
            trtim = np.arange(0,st[0].stats.delta*(st[0].stats.npts),st[0].stats.delta) #-buff
            trtim = trtim[0:trZ.shape[0]]
    
            # Optional: plot seismograms
    
            # fig=plt.figure()
            # ax1=fig.add_subplot(311)
            # ax2=fig.add_subplot(312)
            # ax3=fig.add_subplot(313)
    
            # ax1.plot(trtim,trZ,'k-')
            # ax1.set_ylabel("Z")
            # ax1.set_title(codes[i,1]+','+codes[i,0]+','+codes[i,4]+','+chanstr)
            # ax1.axes.xaxis.set_ticklabels([])
            # ax1.set_xlim(xax)
    
            # ax2.plot(trtim,trN,'k-')
            # ax2.set_ylabel("N")
            # ax2.axes.xaxis.set_ticklabels([])
            # ax2.set_xlim(xax)
    
            # ax3.plot(trtim,trE,'k-')
            # ax3.set_ylabel("E")
            # ax3.set_xlim(xax)
    
            # nom=odir+"/seismograms_%s.png" % (codes[i,0])
            # plt.savefig(nom,dpi=300,bbox_inches='tight')
            # plt.close()
            
    except:
        print('No Data For ',codes[i,1]+'.'+codes[i,0])
        pass


station #0 of 58
UW ALCT -- EN* 2001-02-28T18:54:32.000000Z
UW BEVT -- EN* 2001-02-28T18:54:32.000000Z
UW EARN -- EN* 2001-02-28T18:54:32.000000Z
UW ELW -- EN* 2001-02-28T18:54:32.000000Z
UW GNW -- SN* 2001-02-28T18:54:32.000000Z
UW KIMB -- EN* 2001-02-28T18:54:32.000000Z
UW MBPA -- EN* 2001-02-28T18:54:32.000000Z
UW PCFR -- EN* 2001-02-28T18:54:32.000000Z
UW PNLK -- EN* 2001-02-28T18:54:32.000000Z
UW RAW -- EN* 2001-02-28T18:54:32.000000Z
station #10 of 58
UW RWW -- SN* 2001-02-28T18:54:32.000000Z
UW SEA -- EH* 2001-02-28T18:54:32.000000Z
UW SP2 -- EN* 2001-02-28T18:54:32.000000Z
UW SQM -- SN* 2001-02-28T18:54:32.000000Z
UW TBPA -- EN* 2001-02-28T18:54:32.000000Z
UW TKCO -- EN* 2001-02-28T18:54:32.000000Z
UW TTW -- SN* 2001-02-28T18:54:32.000000Z
UW NOWS -- EN* 2001-02-28T18:54:32.000000Z
UW BRKS -- EN* 2001-02-28T18:54:32.000000Z
UW CSEN -- EN* 2001-02-28T18:54:32.000000Z
station #20 of 58
UW FINN -- EN* 2001-02-28T18:54:32.000000Z
UW LEOT -- EN* 2001-02-28T18:54:32.000000Z
UW KIMR -

In [5]:
pw2sac = '../nisq_data/art_sac_files/'
for i in range(len(sta)):
    try:
        if(codes[i,1]=='GS'):
            timcd = codes[i,4]
            staas = codes[i,0]

            if(i%10==0):
                print('station #'+str(i)+' of '+str(len(sta)))
            
            dt=UTCDateTime(dtstr)

            chanstr=codes[i,3]+'*'
            print(codes[i,1],codes[i,0],codes[i,4],chanstr,dt)
            
            st = read(pw2sac+timcd+'.0000.'+staas+'.ch01.sac',starttime=dt-buff,endtime=dt+120+buff)
            for tr in st:
                tr.data=tr.data/427987 # converts from counts to m/s**2
            st.detrend('demean')
            st.detrend('linear')
            sost = s.butter(2,[flo,fhi],'bandpass',fs=1/st[0].stats.delta,output='sos')
            # sost = s.butter(2,[flo,5],'bandpass',fs=1/st[0].stats.delta,output='sos')
            ST1=s.sosfiltfilt(sost,st[0].data)
            
            st = read(pw2sac+timcd+'.0000.'+staas+'.ch02.sac',starttime=dt-buff,endtime=dt+120+buff)
            for tr in st:
                tr.data=tr.data/427987 # converts from counts to m/s**2
            st.detrend('demean')
            st.detrend('linear')
            sost = s.butter(2,[flo,fhi],'bandpass',fs=1/st[0].stats.delta,output='sos')
            ST2=s.sosfiltfilt(sost,st[0].data)
            
            st = read(pw2sac+timcd+'.0000.'+staas+'.ch03.sac',starttime=dt-buff,endtime=dt+120+buff)
            for tr in st:
                tr.data=tr.data/427987 # converts from counts to m/s**2
            st.detrend('demean')
            st.detrend('linear')
            sost = s.butter(2,[flo,fhi],'bandpass',fs=1/st[0].stats.delta,output='sos')
            ST3=s.sosfiltfilt(sost,st[0].data)
            
            trtim = np.arange(0,st[0].stats.delta*(st[0].stats.npts),st[0].stats.delta) #-buff
            trtim = trtim[0:ST3.shape[0]]
            
            xax=[0.0,120.0] #X-Axis range
    
            # Optional: plot seismograms
            
            # fig=plt.figure()
            # ax1=fig.add_subplot(311)
            # ax2=fig.add_subplot(312)
            # ax3=fig.add_subplot(313)
            
            # ax1.plot(trtim,ST1,'k-')
            # ax1.set_ylabel("Z")
            # ax1.axes.xaxis.set_ticklabels([])
            # ax1.set_xlim(xax)
            
            # ax1.set_title('GS,'+staas+',--,*')
            
            # ax2.plot(trtim,ST2,'k-')
            # ax2.set_ylabel("N")
            # ax2.axes.xaxis.set_ticklabels([])
            # ax2.set_xlim(xax)
            
            # ax3.plot(trtim,ST3,'k-')
            # ax3.set_ylabel("E")
            # ax3.set_xlim(xax)
    
            # nom=odir+"/seismograms_%s.png" % (codes[i,0])
            # plt.savefig(nom,dpi=300,bbox_inches='tight')
            # plt.close()
            
    except:
        print('No Data For ',codes[i,1]+'.'+codes[i,0])
        pass


GS ALK 2001059185424 HN* 2001-02-28T18:54:32.000000Z
GS BOE 2001059185424 HN* 2001-02-28T18:54:32.000000Z
GS BRI 2001059185425 HN* 2001-02-28T18:54:32.000000Z
GS SEW 2001059185425 HN* 2001-02-28T18:54:32.000000Z
GS WEK 2001059185425 HN* 2001-02-28T18:54:32.000000Z
station #40 of 58
GS ALO 2001059185426 HN* 2001-02-28T18:54:32.000000Z
GS BHD 2001059185426 HN* 2001-02-28T18:54:32.000000Z
GS CRO 2001059185426 HN* 2001-02-28T18:54:32.000000Z
GS CTR 2001059185426 HN* 2001-02-28T18:54:32.000000Z
GS EVA 2001059185426 HN* 2001-02-28T18:54:32.000000Z
GS GAR 2001059185426 HN* 2001-02-28T18:54:32.000000Z
GS HAL 2001059185426 HN* 2001-02-28T18:54:32.000000Z
GS HIG 2001059185426 HN* 2001-02-28T18:54:32.000000Z
GS KDK 2001059185426 HN* 2001-02-28T18:54:32.000000Z
GS LAP 2001059185426 HN* 2001-02-28T18:54:32.000000Z
station #50 of 58
GS NOR 2001059185426 HN* 2001-02-28T18:54:32.000000Z
GS PIE 2001059185426 HN* 2001-02-28T18:54:32.000000Z
GS SDN2 2001059185426 HN* 2001-02-28T18:54:32.000000Z
GS SEU 20